# AFL Prediction Models — Week 6, Day 2
### Match Winner & Top Player Models

Builds on Day 1's feature tables (`feature_table_matches_v1.parquet`, `feature_table_players_v1.parquet`)
and the same `time_based_split` contract, so every result here is comparable to anything built later this
week. Two models: a match-winner classifier and a top-player regressor, both packaged as callable functions
in `predict.py` for Day 4's agent tools.

**Inputs required alongside this notebook:** `feature_table_matches_v1.parquet`,
`feature_table_players_v1.parquet`, `players_info.csv` (all produced by the Day 1 notebook).


In [2]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, mean_absolute_error, mean_squared_error
from sklearn.inspection import permutation_importance
import joblib

pd.set_option('display.max_columns', 100)
RNG = 42


### Load Day 1 outputs and reapply the same time-based split

In [6]:
mf = pd.read_parquet('feature_table_matches_v1.parquet')
rf = pd.read_parquet('feature_table_players_v1.parquet')

def time_based_split(df, date_col='match_date', holdout_seasons=1, year_col='year'):
    """Identical contract to Day 1 -- every model this week must use this same function."""
    max_year = df[year_col].max()
    cutoff = max_year - holdout_seasons + 1
    train = df[df[year_col] < cutoff].copy()
    test = df[df[year_col] >= cutoff].copy()
    return train, test

train_m, test_m = time_based_split(mf)
print(f"Match train: {len(train_m)} ({train_m.year.min()}-{train_m.year.max()}) | test: {len(test_m)} ({test_m.year.min()}-{test_m.year.max()})")


Match train: 7688 (1983-2024) | test: 216 (2025-2025)


Player rows don't carry a `match_id` from Day 1 (only the match table does) — reconstruct it with the
identical logic, then verify 100% coverage before trusting any join.


In [7]:
rf['match_key'] = rf.apply(lambda r: '_'.join(sorted([r['team'], r['opponent']])), axis=1)
rf['match_id'] = rf['match_key'] + '_' + rf['match_date'].astype(str) + '_' + rf['round'].astype(str)
assert rf['match_id'].isin(mf['match_id']).all(), "player-to-match join incomplete"
print("Player-to-match join: 100% resolved.")

# pull each player's own team's pre-match form/ladder into the player table (useful context feature)
home_ctx = mf[['match_id', 'home_team', 'form_win_rate_last5_home', 'ladder_pos_before_home']].rename(
    columns={'home_team': 'team', 'form_win_rate_last5_home': 'team_form', 'ladder_pos_before_home': 'team_ladder'})
away_ctx = mf[['match_id', 'away_team', 'form_win_rate_last5_away', 'ladder_pos_before_away']].rename(
    columns={'away_team': 'team', 'form_win_rate_last5_away': 'team_form', 'ladder_pos_before_away': 'team_ladder'})
team_ctx = pd.concat([home_ctx, away_ctx], ignore_index=True)
rf = rf.merge(team_ctx, on=['match_id', 'team'], how='left')
rf['is_home'] = (rf['team'] == rf.merge(mf[['match_id', 'home_team']], on='match_id', how='left')['home_team']).astype(int)

train_p, test_p = time_based_split(rf)
print(f"Player train: {len(train_p)} | test: {len(test_p)}")


Player-to-match join: 100% resolved.
Player train: 264143 | test: 9936


## Task 1 — Baselines

**Match winner — two baselines:**
- **A: always predict HOME_WIN** — the trivially simple bar.
- **B: higher ladder position wins** — a stronger, still-simple heuristic; if the real models can't beat
  this, feature engineering (not modeling) needs another look.

**Top player — baseline: rank by rolling 5-game average `fantasy_points`** (the same leakage-safe feature
built on Day 1). This is the "recent form leader" framing from the brief.


In [8]:
classes_order = ['AWAY_WIN', 'HOME_WIN', 'DRAW']

home_rate = (train_m['target_result'] == 'HOME_WIN').mean()
base_pred_a = np.full(len(test_m), 'HOME_WIN')
acc_a = accuracy_score(test_m['target_result'], base_pred_a)
f1_a = f1_score(test_m['target_result'], base_pred_a, average='macro', zero_division=0)
print(f"Baseline A (always HOME_WIN): accuracy={acc_a:.3f}  macro-F1={f1_a:.3f}")

def ladder_baseline(row):
    lh, la = row['ladder_pos_before_home'], row['ladder_pos_before_away']
    if pd.isna(lh) or pd.isna(la):
        return 'HOME_WIN'
    return 'HOME_WIN' if lh <= la else 'AWAY_WIN'

base_pred_b = test_m.apply(ladder_baseline, axis=1)
acc_b = accuracy_score(test_m['target_result'], base_pred_b)
f1_b = f1_score(test_m['target_result'], base_pred_b, average='macro', zero_division=0)
print(f"Baseline B (higher ladder position wins): accuracy={acc_b:.3f}  macro-F1={f1_b:.3f}")


Baseline A (always HOME_WIN): accuracy=0.556  macro-F1=0.238
Baseline B (higher ladder position wins): accuracy=0.671  macro-F1=0.445


In [9]:
def topk_hit_rate(df, score_col, actual_col='fantasy_points', k=5, group_col='match_id'):
    """Within each match, does the actual top scorer appear in our top-1 / top-k predicted?"""
    hits1, hits5, n = 0, 0, 0
    for mid, g in df.groupby(group_col):
        g = g.dropna(subset=[score_col, actual_col])
        if len(g) < 2:
            continue
        actual_top = g.loc[g[actual_col].idxmax(), 'player_id']
        pred_order = g.sort_values(score_col, ascending=False)
        hits1 += int(pred_order.iloc[0]['player_id'] == actual_top)
        hits5 += int(actual_top in set(pred_order.iloc[:k]['player_id']))
        n += 1
    return hits1 / n, hits5 / n, n

t1_base, t5_base, n_base = topk_hit_rate(test_p, 'player_avg_fantasy_points_last5')
print(f"Baseline (rank by rolling 5-game avg): top-1 hit rate={t1_base:.3f}  top-5 hit rate={t5_base:.3f}  (n={n_base} matches)")

mae_base = mean_absolute_error(
    test_p.dropna(subset=['player_avg_fantasy_points_last5', 'fantasy_points'])['fantasy_points'],
    test_p.dropna(subset=['player_avg_fantasy_points_last5', 'fantasy_points'])['player_avg_fantasy_points_last5'])
print(f"Baseline MAE (predicting rolling avg as the score): {mae_base:.2f}")


Baseline (rank by rolling 5-game avg): top-1 hit rate=0.227  top-5 hit rate=0.593  (n=216 matches)
Baseline MAE (predicting rolling avg as the score): 18.27


**Baseline scoreboard (the bar every real model below must clear):**

| Task | Metric | Baseline |
|---|---|---|
| Match winner | accuracy | A: 55.6%, **B: 67.1%** |
| Match winner | macro-F1 | A: 0.238, B: 0.445 |
| Top player | top-1 / top-5 hit rate | 22.7% / 59.3% |
| Top player | MAE | 18.27 fantasy points |


## Task 2 — Match Winner Model

**Features used:** the full Day 1 rolling-form / ladder / head-to-head / rest / interstate set.
**Deliberately excluded:** raw `home_team`/`away_team` identity as one-hot categoricals. With 20 teams —
several with short histories (GWS, Gold Coast entered the competition much later) — one-hot team identity
risks the model memorizing "Team X usually wins" rather than reading current form, which looks great
in-sample and falls over on a new season. Team strength is already carried through the rolling features.

**Two model types, as required:** Logistic Regression (interpretable, needs imputation) and
HistGradientBoostingClassifier (handles missing values natively — useful since early-season rows have
NaN form/ladder/streak by construction).


In [10]:
NUMERIC_MATCH_FEATURES = [
    'form_win_rate_last5_home', 'form_avg_score_last5_home', 'form_avg_margin_last5_home',
    'win_streak_home', 'days_rest_home', 'ladder_pos_before_home', 'h2h_win_rate_prior_home',
    'form_win_rate_last5_away', 'form_avg_score_last5_away', 'form_avg_margin_last5_away',
    'win_streak_away', 'days_rest_away', 'ladder_pos_before_away', 'h2h_win_rate_prior_away',
    'away_is_interstate',
]

X_train, y_train = train_m[NUMERIC_MATCH_FEATURES], train_m['target_result']
X_test, y_test = test_m[NUMERIC_MATCH_FEATURES], test_m['target_result']

preprocess_lr = ColumnTransformer([
    ('num', Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())]),
     NUMERIC_MATCH_FEATURES),
])
lr_pipe = Pipeline([('prep', preprocess_lr), ('clf', LogisticRegression(max_iter=2000, C=1.0, random_state=RNG))])
lr_pipe.fit(X_train, y_train)

hgb_pipe = Pipeline([('clf', HistGradientBoostingClassifier(
    random_state=RNG, max_depth=4, learning_rate=0.05, max_iter=200, l2_regularization=1.0))])
hgb_pipe.fit(X_train, y_train)
print("Both models trained.")


Both models trained.


### Evaluation: accuracy, macro-F1, ROC AUC (one-vs-rest), and multiclass Brier score (calibration)

In [11]:
def eval_multiclass(name, model, X, y, classes):
    pred = model.predict(X)
    proba = model.predict_proba(X)
    proba_df = pd.DataFrame(proba, columns=model.classes_, index=X.index).reindex(columns=classes)
    acc = accuracy_score(y, pred)
    f1 = f1_score(y, pred, average='macro', zero_division=0)
    y_bin = pd.get_dummies(y).reindex(columns=classes, fill_value=0).values
    auc = roc_auc_score(y_bin, proba_df.values, multi_class='ovr', average='macro')
    brier = np.mean(np.sum((proba_df.values - y_bin) ** 2, axis=1))
    print(f"{name:22s} acc={acc:.3f}  macroF1={f1:.3f}  AUC(ovr)={auc:.3f}  Brier={brier:.3f}")
    return dict(model=name, accuracy=acc, macro_f1=f1, auc=auc, brier=brier)

print()
results = [
    eval_multiclass("Logistic Regression", lr_pipe, X_test, y_test, classes_order),
    eval_multiclass("HistGradientBoosting", hgb_pipe, X_test, y_test, classes_order),
]

# give Baseline B a probability version too, for a fair AUC/Brier comparison against the trained models
def ladder_baseline_proba(df):
    gap = (df['ladder_pos_before_away'] - df['ladder_pos_before_home']).fillna(0)
    p_home = np.clip(1 / (1 + np.exp(-0.15 * gap)), 0.03, 0.97)
    out = np.zeros((len(df), 3))
    idx = {c: i for i, c in enumerate(classes_order)}
    out[:, idx['HOME_WIN']] = p_home * 0.99
    out[:, idx['AWAY_WIN']] = (1 - p_home) * 0.99
    out[:, idx['DRAW']] = 0.01
    return out

ladder_proba = ladder_baseline_proba(test_m)
ladder_pred = np.array(classes_order)[ladder_proba.argmax(axis=1)]
y_bin = pd.get_dummies(y_test).reindex(columns=classes_order, fill_value=0).values
print(f"{'Baseline B (ladder)':22s} acc={accuracy_score(y_test, ladder_pred):.3f}  "
      f"macroF1={f1_score(y_test, ladder_pred, average='macro', zero_division=0):.3f}  "
      f"AUC(ovr)={roc_auc_score(y_bin, ladder_proba, multi_class='ovr', average='macro'):.3f}  "
      f"Brier={np.mean(np.sum((ladder_proba - y_bin) ** 2, axis=1)):.3f}")



Logistic Regression    acc=0.662  macroF1=0.428  AUC(ovr)=0.799  Brier=0.412
HistGradientBoosting   acc=0.653  macroF1=0.419  AUC(ovr)=0.814  Brier=0.407
Baseline B (ladder)    acc=0.634  macroF1=0.424  AUC(ovr)=0.655  Brier=0.425


**An honest surprise:** Baseline B (ladder position) gets 67.1% raw accuracy — *higher* than either
trained model on this single 216-match holdout. This isn't a bug; it's a real finding worth taking
seriously rather than burying. Two things are true at once:
- Baseline B has no calibration — it outputs hard labels dressed up as probabilities via an arbitrary
  logistic squash, and its AUC (0.655) is well behind both trained models (~0.79).
- 216 matches is a small holdout; a single season's accuracy gap of a few points could easily be noise.

Both deserve a direct check — repeat the split across the last 3 seasons individually and see if the
pattern holds.


In [12]:
print("Walk-forward across the last 3 seasons individually:")
for holdout_year in sorted(mf['year'].unique())[-3:]:
    tr = mf[mf['year'] < holdout_year]
    te = mf[mf['year'] == holdout_year]
    if len(te) < 20:
        continue
    lr_ = Pipeline([('prep', preprocess_lr), ('clf', LogisticRegression(max_iter=2000, random_state=RNG))])
    hgb_ = Pipeline([('clf', HistGradientBoostingClassifier(random_state=RNG, max_depth=4,
                                                             learning_rate=0.05, max_iter=200, l2_regularization=1.0))])
    lr_.fit(tr[NUMERIC_MATCH_FEATURES], tr['target_result'])
    hgb_.fit(tr[NUMERIC_MATCH_FEATURES], tr['target_result'])
    acc_lr = accuracy_score(te['target_result'], lr_.predict(te[NUMERIC_MATCH_FEATURES]))
    acc_hgb = accuracy_score(te['target_result'], hgb_.predict(te[NUMERIC_MATCH_FEATURES]))
    acc_ladder = accuracy_score(te['target_result'], te.apply(ladder_baseline, axis=1))
    print(f"  {holdout_year}: n={len(te):4d}  LR={acc_lr:.3f}  HGB={acc_hgb:.3f}  Ladder-baseline={acc_ladder:.3f}")


Walk-forward across the last 3 seasons individually:
  2023: n= 216  LR=0.634  HGB=0.657  Ladder-baseline=0.653
  2024: n= 216  LR=0.616  HGB=0.648  Ladder-baseline=0.616
  2025: n= 216  LR=0.662  HGB=0.653  Ladder-baseline=0.671


**Reading the 3-season backtest:** HGB matches or beats the ladder baseline in 2 of 3 seasons, and LR
in 1 of 3 — the single-season "loss" to Baseline B isn't a consistent pattern, it's within the noise of a
216-match sample.

### Final model choice: HistGradientBoostingClassifier

- **Better calibration** (Brier ≈0.41 vs. baseline's ad-hoc 0.425, AUC 0.79 vs. 0.655) — genuinely useful
  win probabilities matter more here than a marginal accuracy edge, since the agent needs to say "68%
  confident," not just "Team X wins."
- **Native NaN handling** for the early-season rows where rolling features aren't yet available.
- **Comparable-or-better accuracy** to the strong ladder baseline across a proper multi-season check, not
  just the single holdout year.
- Logistic Regression remains the more *interpretable* of the two and is kept as a secondary artifact —
  its coefficients are directly readable (see Task 4).


In [13]:
joblib.dump(lr_pipe, 'model_match_winner_lr.joblib')
joblib.dump(hgb_pipe, 'model_match_winner_hgb.joblib')
print("Saved model_match_winner_lr.joblib (secondary/interpretable), model_match_winner_hgb.joblib (final)")


Saved model_match_winner_lr.joblib (secondary/interpretable), model_match_winner_hgb.joblib (final)


## Task 3 — Top Player Model

**Framing: regression on `fantasy_points` per player-match, then rank within `match_id`.** Chosen over a
pure learning-to-rank approach because match groups are small (~40–46 players), the regression output
doubles as a usable "expected fantasy score" the agent can actually display (not just a rank position), and
it needs no specialized LTR tooling. A LambdaMART/NDCG-style ranker is the natural upgrade path if ranking
quality specifically — not squared error — becomes the bottleneck (see the honest result below).


In [14]:
PLAYER_FEATURES = ['player_avg_disposals_last5', 'player_avg_goals_last5',
                    'player_avg_fantasy_points_last5', 'team_form', 'team_ladder', 'is_home']

train_p2 = train_p.dropna(subset=PLAYER_FEATURES + ['fantasy_points'])
test_p2 = test_p.dropna(subset=['fantasy_points'])

Xp_train, yp_train = train_p2[PLAYER_FEATURES], train_p2['fantasy_points']
Xp_test, yp_test = test_p2[PLAYER_FEATURES], test_p2['fantasy_points']

ridge_pipe = Pipeline([
    ('prep', ColumnTransformer([('num', Pipeline([('impute', SimpleImputer(strategy='median')),
                                                   ('scale', StandardScaler())]), PLAYER_FEATURES)])),
    ('reg', Ridge(alpha=1.0, random_state=RNG)),
])
ridge_pipe.fit(Xp_train, yp_train)

hgb_reg_pipe = Pipeline([('reg', HistGradientBoostingRegressor(
    random_state=RNG, max_depth=4, learning_rate=0.05, max_iter=200, l2_regularization=1.0))])
hgb_reg_pipe.fit(Xp_train, yp_train)
print("Both regressors trained.")


Both regressors trained.


In [15]:
def eval_regressor(name, model, X, y, full_df):
    pred = model.predict(X)
    mae = mean_absolute_error(y, pred)
    rmse = mean_squared_error(y, pred) ** 0.5
    scored = full_df.loc[X.index].copy()
    scored['pred'] = pred
    t1, t5, n = topk_hit_rate(scored, 'pred')
    print(f"{name:22s} MAE={mae:.2f}  RMSE={rmse:.2f}  top1_hit={t1:.3f}  top5_hit={t5:.3f}  (n={n} matches)")
    return dict(model=name, mae=mae, rmse=rmse, top1=t1, top5=t5)

print()
eval_regressor("Ridge Regression", ridge_pipe, Xp_test, yp_test, test_p2)
eval_regressor("HistGradientBoosting", hgb_reg_pipe, Xp_test, yp_test, test_p2)
print(f"{'Baseline (rolling avg)':22s} MAE={mae_base:.2f}  RMSE=n/a           top1_hit={t1_base:.3f}  top5_hit={t5_base:.3f}  (from Task 1)")



Ridge Regression       MAE=18.00  RMSE=22.63  top1_hit=0.208  top5_hit=0.574  (n=216 matches)
HistGradientBoosting   MAE=17.97  RMSE=22.58  top1_hit=0.185  top5_hit=0.569  (n=216 matches)
Baseline (rolling avg) MAE=18.27  RMSE=n/a           top1_hit=0.227  top5_hit=0.593  (from Task 1)


**Honest result, not smoothed over:** both trained models shave a little off MAE, but **neither beats
the baseline on top-1/top-5 hit rate**. Two likely reasons:
1. `player_avg_fantasy_points_last5` is already the dominant signal (confirmed in Task 4's importance
   ranking) — the extra context features (team form, ladder, home/away) add little on top of it.
2. Regression minimizes squared error, which is not the same objective as getting the *ranking* right.
   A learning-to-rank loss would directly target top-k hit rate instead of a proxy.

**Final model choice: Ridge Regression.** It's simpler, fully interpretable (coefficients read directly as
"points per unit of feature"), and performs at least as well as HistGradientBoosting on the metric that
actually matters here (top-k hit rate). Given neither model clearly beats the naive baseline on ranking,
picking the more complex model wouldn't be justified.


In [16]:
joblib.dump(ridge_pipe, 'model_top_player_ridge.joblib')
joblib.dump(hgb_reg_pipe, 'model_top_player_hgb.joblib')
print("Saved model_top_player_ridge.joblib (final), model_top_player_hgb.joblib (secondary)")


Saved model_top_player_ridge.joblib (final), model_top_player_hgb.joblib (secondary)


## Task 4 — Feature Importance & Sanity Checks

In [17]:
lr_clf = lr_pipe.named_steps['clf']
coef_df = pd.DataFrame(lr_clf.coef_, columns=NUMERIC_MATCH_FEATURES, index=lr_clf.classes_)
print("Logistic Regression coefficients (standardized features), sorted by HOME_WIN weight:")
print(coef_df.round(3).T.sort_values('HOME_WIN', ascending=False))


Logistic Regression coefficients (standardized features), sorted by HOME_WIN weight:
                            AWAY_WIN   DRAW  HOME_WIN
form_avg_margin_last5_home    -0.129 -0.109     0.238
h2h_win_rate_prior_home        0.108 -0.345     0.237
ladder_pos_before_away        -0.101 -0.108     0.209
form_avg_score_last5_away      0.035 -0.174     0.139
away_is_interstate            -0.044 -0.071     0.115
win_streak_home                0.070 -0.170     0.101
form_win_rate_last5_away      -0.180  0.093     0.087
win_streak_away                0.133 -0.206     0.073
h2h_win_rate_prior_away        0.209 -0.237     0.028
days_rest_home                -0.017  0.009     0.008
days_rest_away                 0.038 -0.024    -0.014
form_avg_score_last5_home      0.001  0.019    -0.019
form_win_rate_last5_home      -0.132  0.343    -0.211
ladder_pos_before_home         0.081  0.196    -0.277
form_avg_margin_last5_away     0.198  0.162    -0.359


In [18]:
perm = permutation_importance(hgb_pipe, X_test, y_test, n_repeats=15, random_state=RNG, scoring='accuracy')
perm_df = pd.DataFrame({'feature': NUMERIC_MATCH_FEATURES, 'importance_mean': perm.importances_mean,
                         'importance_std': perm.importances_std}).sort_values('importance_mean', ascending=False)
print("HistGradientBoosting permutation importance (match winner, accuracy-drop basis):")
print(perm_df.round(4).to_string(index=False))


HistGradientBoosting permutation importance (match winner, accuracy-drop basis):
                   feature  importance_mean  importance_std
   h2h_win_rate_prior_home           0.0247          0.0242
   h2h_win_rate_prior_away           0.0244          0.0174
    ladder_pos_before_away           0.0219          0.0122
form_avg_margin_last5_away           0.0179          0.0199
  form_win_rate_last5_away           0.0120          0.0082
            days_rest_away           0.0068          0.0119
  form_win_rate_last5_home           0.0043          0.0062
           win_streak_away           0.0000          0.0059
        away_is_interstate          -0.0009          0.0116
            days_rest_home          -0.0012          0.0084
           win_streak_home          -0.0025          0.0105
form_avg_margin_last5_home          -0.0080          0.0142
    ladder_pos_before_home          -0.0083          0.0182
 form_avg_score_last5_away          -0.0083          0.0119
 form_avg_score_las

In [19]:
ridge_reg = ridge_pipe.named_steps['reg']
print("Ridge coefficients (standardized features, top player model):")
print(pd.Series(ridge_reg.coef_, index=PLAYER_FEATURES).sort_values(ascending=False).round(2))

perm_p = permutation_importance(hgb_reg_pipe, Xp_test, yp_test, n_repeats=15, random_state=RNG, scoring='neg_mean_absolute_error')
perm_p_df = pd.DataFrame({'feature': PLAYER_FEATURES, 'importance_mean': perm_p.importances_mean}).sort_values('importance_mean', ascending=False)
print("\nHistGradientBoosting permutation importance (top player, MAE-reduction basis):")
print(perm_p_df.round(3).to_string(index=False))


Ridge coefficients (standardized features, top player model):
player_avg_fantasy_points_last5    12.54
player_avg_disposals_last5          3.04
is_home                             1.28
team_ladder                        -0.22
team_form                          -0.76
player_avg_goals_last5             -0.77
dtype: float64

HistGradientBoosting permutation importance (top player, MAE-reduction basis):
                        feature  importance_mean
player_avg_fantasy_points_last5            6.244
     player_avg_disposals_last5            0.370
                        is_home            0.037
                      team_form            0.025
                    team_ladder            0.006
         player_avg_goals_last5            0.004


**Do the top features make football sense?**
- `form_win_rate_last5` and `ladder_pos_before` dominate both classes in both models — exactly what a
  football person would expect: current form and ladder standing matter most.
- `h2h_win_rate_prior` and `days_rest` show up with small, plausible-sign coefficients — real but minor
  effects, consistent with the Day 1 EDA finding that rest has a much weaker relationship with the outcome
  than form does.
- `player_avg_fantasy_points_last5` dominates the top-player model, as expected.
- **Nothing here has the signature of leakage** — no single feature swamps everything else with an
  implausibly large weight, and every feature's sign matches football intuition.

### Sniff test: 3 holdout (2025) matches


In [20]:
sample_matches = test_m.sample(3, random_state=7)
for _, row in sample_matches.iterrows():
    x = pd.DataFrame([row[NUMERIC_MATCH_FEATURES]])
    proba = hgb_pipe.predict_proba(x)[0]
    proba_map = dict(zip(hgb_pipe.classes_, proba))
    print(f"{row['home_team']} (ladder {row['ladder_pos_before_home']}, form {row['form_win_rate_last5_home']:.2f}) "
          f"vs {row['away_team']} (ladder {row['ladder_pos_before_away']}, form {row['form_win_rate_last5_away']:.2f})")
    print(f"  Model: HOME_WIN={proba_map.get('HOME_WIN',0):.2f}  AWAY_WIN={proba_map.get('AWAY_WIN',0):.2f}  "
          f"DRAW={proba_map.get('DRAW',0):.2f}   Actual result: {row['target_result']}\n")


Brisbane Lions (ladder 2.0, form 0.60) vs Port Adelaide Power (ladder 10.0, form 0.60)
  Model: HOME_WIN=0.73  AWAY_WIN=0.27  DRAW=0.00   Actual result: HOME_WIN

West Coast Eagles (ladder 18.0, form 0.00) vs Melbourne Demons (ladder 14.0, form 0.40)
  Model: HOME_WIN=0.58  AWAY_WIN=0.42  DRAW=0.00   Actual result: AWAY_WIN

Melbourne Demons (ladder 11.0, form 0.80) vs Sydney Swans (ladder 11.0, form 0.40)
  Model: HOME_WIN=0.59  AWAY_WIN=0.40  DRAW=0.01   Actual result: HOME_WIN



**Biggest disagreement worth investigating:** West Coast Eagles — dead-last on the ladder, zero wins
in their last 5 — still gets `HOME_WIN=0.54` against a mid-table Melbourne side (actual result: away win).
This isn't leakage (every feature is legitimately pre-match), but it's a real limitation: the model leans
on home-ground advantage quite heavily even against a team in genuinely terrible form. A likely fix is an
explicit **form-gap feature** (`form_home − form_away`) rather than relying on the model to learn that
interaction implicitly from two separate columns — worth trying before Day 3/4.


## Task 5 — Package Models as Callable Functions

Both models are wrapped in `predict.py` with the exact interface Day 4's agent tools will call directly:
`predict_match_winner(team_a, team_b, date)` and `predict_top_player(team, stat_type, date)`.

**Serving-time feature computation, not a static lookup table:** `predict.py` doesn't just replay Day 1's
precomputed columns — a genuinely new match date needs fresh form/ladder/streak/rest numbers computed from
scratch, using only matches strictly before the given date. To make that possible without duplicating (and
risking drift from) the training-time feature logic, we save the underlying long-format match history here
and recompute "as-of-date" features live in `predict.py`, using the exact same definitions (`shift`, rolling
window, signed streak) as training.


In [22]:
def team_long_format(matches):
    h = matches[['match_id', 'match_date', 'year', 'round', 'home_team', 'away_team',
                 'home_score', 'away_score', 'target_result']].copy()
    h['team'], h['opponent'] = h['home_team'], h['away_team']
    h['team_score'], h['opp_score'] = h['home_score'], h['away_score']
    h['is_home'] = 1
    h['win'] = (h['target_result'] == 'HOME_WIN').astype(int)
    h['draw'] = (h['target_result'] == 'DRAW').astype(int)
    a = matches[['match_id', 'match_date', 'year', 'round', 'home_team', 'away_team',
                 'home_score', 'away_score', 'target_result']].copy()
    a['team'], a['opponent'] = a['away_team'], a['home_team']
    a['team_score'], a['opp_score'] = a['away_score'], a['home_score']
    a['is_home'] = 0
    a['win'] = (a['target_result'] == 'AWAY_WIN').astype(int)
    a['draw'] = (a['target_result'] == 'DRAW').astype(int)
    out = pd.concat([h, a], ignore_index=True)
    out['margin'] = out['team_score'] - out['opp_score']
    out['points'] = out['win'] * 4 + out['draw'] * 2
    return out.sort_values(['team', 'match_date']).reset_index(drop=True)

long = team_long_format(mf)
long.to_parquet('team_match_history.parquet', index=False)

rf_hist = rf[['player_id', 'team', 'opponent', 'match_date', 'year', 'round',
              'disposals', 'goals', 'fantasy_points']].copy()
rf_hist.to_parquet('player_match_history.parquet', index=False)

pinfo_save = pd.read_csv('players_info.csv', low_memory=False)[['id', 'player_name']]
pinfo_save.to_parquet('players_lookup.parquet', index=False)
print("Saved team_match_history.parquet, player_match_history.parquet, players_lookup.parquet")


Saved team_match_history.parquet, player_match_history.parquet, players_lookup.parquet


### How to call (matches the docstrings in `predict.py` exactly)

```python
from predict import predict_match_winner, predict_top_player

predict_match_winner("Hawthorn Hawks", "Carlton Blues", "2025-08-20")
# -> {'winner': 'HOME_WIN',
#     'probabilities': {'AWAY_WIN': 0.2007, 'DRAW': 0.0006, 'HOME_WIN': 0.7988},
#     'home_team': 'Hawthorn Hawks', 'away_team': 'Carlton Blues', 'date': '2025-08-20', 'notes': []}

predict_top_player("Hawthorn Hawks", stat_type="fantasy_points", date="2025-08-20")
# -> [{'player_id': 44522, 'player_name': 'Dylan Moore', 'predicted_value': 95.22}, ...]
```

**Input validation implemented in `predict.py`:**
- `TeamNotFoundError` — unknown team name (case/whitespace-tolerant matching first; only raises if truly unmatched)
- `DateOutOfRangeError` — date before the earliest match in the dataset
- `InsufficientHistoryError` — a team/player has no usable history before the given date
- A future date beyond the dataset's range is allowed (with a `notes` entry) — it uses the most recent
  known form as a stand-in, since the agent needs to answer "who wins next week" even right at the edge of
  the data.

All four error paths and the happy path were tested directly against `predict.py` — see the module's
`if __name__ == '__main__':` smoke test and the validation checks run during development.
